# Taxas Relacionadas — Aulão de Monitoria (Cálculo 1)

**Ideia da aula:** taxas relacionadas não são um "truque de prova", são o jeito de a matemática responder a perguntas do tipo *"se esta grandeza muda assim, com que rapidez aquela outra muda?"* — e isso aparece em óptica, robótica, astronomia, engenharia espacial etc.

| # | Problema | Formato | Onde isso aparece no mundo real |
|---|----------|---------|----------------------------------|
| 1 | Lente delgada | 📋 quadro | foco de câmeras e telescópios |
| 2 | Braço articulado | ✏️ alunos | cinemática de robôs e mecanismos articulados |
| 3 | Cometa e lei das áreas | 📋 quadro | mecânica celeste (volta ao gancho histórico da aula) |
| 4 | Decaimento orbital | ✏️ alunos | satélites em órbita baixa e arrasto atmosférico |

A ordem é de dificuldade crescente e alterna **"eu faço" / "vocês fazem"**.

**Como usar este notebook**

- Rode as células de cima para baixo. Sliders e animações **não precisam de edição de código**.
- Nos problemas, a figura começa **sem a resposta**. Marque **conferir** para revelar o gráfico da resposta (útil depois de resolver no quadro).
- Dicas e resoluções ficam em blocos recolhíveis (`▶ Resolução…`): clique só quando quiser mostrar.
- As animações aparecem ao clicar em **Run Interact** (não rodam a cada movimento de slider, para não travar o telão).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Arc
from IPython.display import HTML, display
from ipywidgets import interact, interact_manual, FloatSlider

plt.rcParams.update({
    "figure.dpi": 100,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Paleta usada em todo o notebook
AZUL, VERMELHO, LARANJA, CINZA, VERDE = "#2c7fb8", "#d7301f", "#f5a623", "#444444", "#238b45"


def br(x, nd=2):
    """Número com vírgula decimal (padrão brasileiro): br(1.25) -> '1,25'."""
    return f"{x:.{nd}f}".replace(".", ",").replace("-", "−")


def derivada(f, t, h=1e-6):
    """Derivada numérica por diferença central: (f(t+h) - f(t-h)) / (2h).
    É a definição de derivada, só que sem tomar o limite."""
    return (f(t + h) - f(t - h)) / (2 * h)


def mostrar_animacao(fig, update, n_frames, interval=60):
    """Figura + função update(i) -> animação com botões play/pause (JavaScript, sem ffmpeg)."""
    anim = animation.FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=False)
    plt.close(fig)  # evita mostrar a figura estática duplicada
    display(HTML(anim.to_jshtml()))

---
## Parte 1 — Motivação histórica: Kepler, Newton e a lei das áreas

Johannes Kepler, analisando as observações de Tycho Brahe sobre Marte, publicou em 1609 (*Astronomia Nova*) a regra que hoje chamamos de **Segunda Lei de Kepler**:

> O raio que liga um planeta ao Sol varre áreas iguais em tempos iguais.

Era uma regra *observada*: funcionava, mas ninguém sabia **por quê**. Em 1687, no *Principia* (Livro I, Proposição 1), Newton demonstrou que a lei vale para **qualquer força central** — com argumentos geométricos baseados em triângulos cada vez menores, ou seja, em passagens ao limite: a ideia que dá origem ao cálculo.

Essa frase já esconde uma taxa relacionada:

$$\frac{dA}{dt} = \text{constante}$$

onde $A$ é a área varrida pelo raio vetor (a linha que liga o planeta ao Sol). É exatamente esse tipo de relação — uma grandeza geométrica variando com o tempo, ligada a outra por uma equação — que vocês vão praticar hoje, só que com **lentes, braços articulados, cometas e satélites**.

In [2]:
def solve_kepler(M, e, tol=1e-10, max_iter=100):
    """Resolve a equação de Kepler M = E - e*sin(E) para a anomalia excêntrica E,
    via Newton-Raphson. M pode ser um array (um valor de M por instante de tempo)."""
    M = np.asarray(M, dtype=float)
    E = np.where(e < 0.8, M, np.pi * np.ones_like(M))
    for _ in range(max_iter):
        f = E - e * np.sin(E) - M
        fp = 1 - e * np.cos(E)
        dE = f / fp
        E = E - dE
        if np.max(np.abs(dE)) < tol:
            break
    return E


def orbit_xy(t, a, e, T):
    """Posição (x, y) do planeta no instante t, com o Sol (foco da elipse) na origem.
    a = semieixo maior, e = excentricidade, T = período orbital."""
    n = 2 * np.pi / T  # movimento médio (velocidade angular média)
    M = n * t  # anomalia média
    E = solve_kepler(M, e)  # anomalia excêntrica
    x = a * (np.cos(E) - e)
    y = a * np.sqrt(1 - e**2) * np.sin(E)
    return x, y


def swept_area(t0, t1, a, e, T, n_steps=1500):
    """Área varrida pelo raio vetor (Sol -> planeta) entre os instantes t0 e t1,
    calculada numericamente (fórmula do polígono / shoelace)."""
    ts = np.linspace(t0, t1, n_steps)
    x, y = orbit_xy(ts, a, e, T)
    x = np.concatenate([[0.0], x])
    y = np.concatenate([[0.0], y])
    area = 0.5 * np.abs(np.sum(x[:-1] * y[1:] - x[1:] * y[:-1]))
    return area

### Demonstração 1 — Lei das áreas iguais (interativa)

Mexa na excentricidade $e$ abaixo: mesmo quando a órbita fica bem "achatada" (mais elíptica), os dois setores — perto do periélio (ponto mais próximo do Sol, em azul) e perto do afélio (ponto mais distante, em vermelho) — continuam com **a mesma área**, desde que o $\Delta t$ seja o mesmo. Repare como o planeta precisa se mover muito mais rápido perto do periélio para "compensar" o raio menor.

In [ ]:
def plot_equal_areas(e=0.6, dt=0.12, a=1.0, T=1.0):
    fig, ax = plt.subplots(figsize=(6, 6))

    theta = np.linspace(0, 2 * np.pi, 400)
    x_orb = a * (np.cos(theta) - e)
    y_orb = a * np.sqrt(1 - e**2) * np.sin(theta)
    ax.plot(x_orb, y_orb, color=CINZA, lw=1.5)
    ax.plot(0, 0, "o", color=LARANJA, markersize=16, label="Sol (foco)")

    setores = [(0.0, AZUL, "perto do periélio"), (T / 2, VERMELHO, "perto do afélio")]
    for t0, color, label in setores:
        ts = np.linspace(t0, t0 + dt, 60)
        xs, ys = orbit_xy(ts, a, e, T)
        xs_poly = np.concatenate([[0.0], xs])
        ys_poly = np.concatenate([[0.0], ys])
        ax.fill(xs_poly, ys_poly, color=color, alpha=0.55, label=f"setor {label}")

    A1 = swept_area(0.0, dt, a, e, T)
    A2 = swept_area(T / 2, T / 2 + dt, a, e, T)
    ax.set_aspect("equal")
    ax.grid(False)
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(
        f"Área perto do periélio = {A1:.4f}   |   perto do afélio = {A2:.4f}\n"
        f"(diferença ≈ {abs(A1 - A2):.1e})",
        fontsize=10,
    )
    plt.show()


interact(
    plot_equal_areas,
    e=FloatSlider(min=0.0, max=0.9, step=0.05, value=0.6, description="excentricidade e"),
    dt=FloatSlider(min=0.02, max=0.3, step=0.02, value=0.12, description="Δt"),
);

interactive(children=(FloatSlider(value=0.6, description='excentricidade e', max=0.9, step=0.05), FloatSlider(…

### Demonstração 2 — Animação de um período orbital completo

Essa célula gera uma animação do planeta completando uma órbita inteira — boa para deixar rodando no telão enquanto você fala. Ajuste a excentricidade e clique em **"Run Interact"** para gerar de novo (a animação não é recalculada a cada movimento do slider, só quando você manda — assim não trava).

In [4]:
def animar_orbita(e=0.6, n_frames=90, a=1.0, T=1.0):
    t_vals = np.linspace(0, T, n_frames, endpoint=False)
    xs, ys = orbit_xy(t_vals, a, e, T)

    theta_full = np.linspace(0, 2 * np.pi, 300)
    x_orb = a * (np.cos(theta_full) - e)
    y_orb = a * np.sqrt(1 - e**2) * np.sin(theta_full)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(x_orb, y_orb, color=CINZA, lw=1)
    ax.plot(0, 0, "o", color=LARANJA, markersize=14)
    ax.set_aspect("equal")
    ax.grid(False)
    margin = 0.2
    ax.set_xlim(x_orb.min() - margin, x_orb.max() + margin)
    ax.set_ylim(y_orb.min() - margin, y_orb.max() + margin)
    ax.set_title(f"Órbita completa — e = {e:.2f}")

    planet, = ax.plot([], [], "o", color=AZUL, markersize=10)
    raio, = ax.plot([], [], "-", color=AZUL, lw=1, alpha=0.6)

    def init():
        planet.set_data([], [])
        raio.set_data([], [])
        return planet, raio

    def update(i):
        planet.set_data([xs[i]], [ys[i]])
        raio.set_data([0, xs[i]], [0, ys[i]])
        return planet, raio

    anim = animation.FuncAnimation(
        fig, update, frames=n_frames, init_func=init, blit=True, interval=40
    )
    plt.close(fig)  # evita mostrar a figura estática duplicada
    display(HTML(anim.to_jshtml()))


interact_manual(
    animar_orbita,
    e=FloatSlider(min=0.0, max=0.9, step=0.05, value=0.6, description="excentricidade e"),
);

interactive(children=(FloatSlider(value=0.6, description='excentricidade e', max=0.9, step=0.05), IntSlider(va…

### A ponte para hoje: de Kepler para Taxas Relacionadas

Em coordenadas polares, a área de um setor bem fino, de ângulo $d\theta$ e raio $r$, é aproximadamente

$$dA = \frac{1}{2} r^2\, d\theta$$

Dividindo os dois lados por $dt$:

$$\frac{dA}{dt} = \frac{1}{2} r^2 \frac{d\theta}{dt}$$

Newton mostrou que essa quantidade é **constante** ao longo de toda a órbita — e essa equação, com uma grandeza (a área varrida) e outra ($\theta$, o ângulo) mudando juntas no tempo, ligadas por uma equação, **é exatamente uma taxa relacionada**. Só que aplicada a planetas, em vez de escadas ou balões.

Repare no que isso diz sobre $r$ e $\dot\theta$: se $r^2\dot\theta$ é constante, então **quando $r$ cresce, $\dot\theta$ tem que diminuir**. Um planeta (ou cometa) gira mais devagar ao se afastar do Sol. Vamos usar isso no Problema 3.

---
## Parte 2 — A ideia central e o método

Se duas grandezas estão ligadas por uma equação, **as suas taxas de variação também ficam ligadas**. Exemplo mínimo: se $A = \pi r^2$ e o raio cresce com o tempo, então

$$\frac{dA}{dt} = 2\pi r\,\frac{dr}{dt}$$

(regra da cadeia: $A$ depende de $r$, e $r$ depende de $t$).

### Receita em 5 passos

1. **Nomeie** as grandezas que variam com $t$ e anote o que é constante. Faça um desenho.
2. **Escreva uma equação** que relacione as grandezas **em todo instante**.
3. **Derive os dois lados em relação a $t$** (regra da cadeia em cada variável).
4. **Só agora substitua** os valores do instante pedido (e calcule o que faltar, usando a equação original).
5. **Interprete**: o sinal (aumenta ou diminui?), a unidade e se o número faz sentido.

> ⚠️ **Erro mais comum:** substituir os números *antes* de derivar. Isso transforma uma variável em constante e a derivada some. Ex.: na lente, se você trocar $p$ por $60$ logo no começo, "$q$ fica fixo" e $q'=0$ — errado. $p$ e $q$ **variam**; só congelamos o instante depois de derivar.

---
## Parte 3 — Os quatro problemas

### Problema 1 — Lente delgada &nbsp; 📋 *(no quadro)*

Uma lente tem $f = 20$ cm. Um objeto se aproxima dela a $5$ cm/s.

- **(a)** Com que rapidez a imagem se move quando $p = 60$ cm? Ela se aproxima ou se afasta?
- **(b)** Em que $p$ a imagem e o objeto têm a mesma rapidez?
- **(c)** O que acontece com a rapidez da imagem quando $p \to f$?

*Contexto:* é a equação que está por trás do foco de câmeras, óculos e telescópios. Aqui $p$ é a distância objeto–lente e $q$ a distância imagem–lente, ligadas por $\dfrac1p+\dfrac1q=\dfrac1f$.

**Na figura:** arraste $p$ e veja a imagem (em vermelho) se mover; os raios laranja são os dois raios principais. Depois de resolver, marque **conferir**.

In [5]:
F_LENTE = 20.0   # distância focal (cm)
V_OBJ = -5.0     # p' (cm/s): negativo = objeto se aproximando


def q_lente(p, f=F_LENTE):
    """Posição da imagem pela equação da lente: 1/p + 1/q = 1/f."""
    return f * p / (p - f)


def desenhar_lente(axs, p, conferir=False, f=F_LENTE, pdot=V_OBJ):
    ax1, ax2 = axs
    ax1.clear()
    ax2.clear()
    q = q_lente(p, f)
    ho = 10.0          # altura (arbitrária) do objeto
    hi = ho * q / p    # altura da imagem (invertida)

    # --- esquema óptico ---
    ax1.axhline(0, color=CINZA, lw=1)
    ax1.annotate("", xy=(0, 38), xytext=(0, -38),
                 arrowprops=dict(arrowstyle="<->", color=CINZA, lw=2))
    for xf, nome in ((-f, "F"), (f, "F′")):
        ax1.plot(xf, 0, "o", color=CINZA, ms=5)
        ax1.text(xf, -3, nome, ha="center", va="top")
    ax1.annotate("", xy=(-p, ho), xytext=(-p, 0),
                 arrowprops=dict(arrowstyle="-|>", color=AZUL, lw=3, mutation_scale=15))
    ax1.annotate("", xy=(q, -hi), xytext=(q, 0),
                 arrowprops=dict(arrowstyle="-|>", color=VERMELHO, lw=3, mutation_scale=15))
    ax1.plot([-p, 0, q], [ho, ho, -hi], color=LARANJA, lw=1.2)  # raio paralelo -> passa pelo foco
    ax1.plot([-p, q], [ho, -hi], color=LARANJA, lw=1.2)         # raio pelo centro da lente
    ax1.text(-p, ho + 2, f"objeto\np = {br(p, 0)} cm", ha="center", color=AZUL)
    ax1.text(q, -hi - 3, f"imagem\nq = {br(q, 1)} cm", ha="center", va="top", color=VERMELHO)
    ax1.set_xlim(-110, 110)
    ax1.set_ylim(-60, 48)
    ax1.set_yticks([])
    ax1.set_xlabel("posição ao longo do eixo (cm)")
    ax1.set_title("1/p + 1/q = 1/f   (f = 20 cm)")

    # --- gráfico da resposta (só com "conferir") ---
    ax2.set_xlabel("p (cm)")
    if not conferir:
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.grid(False)
        ax2.text(0.5, 0.5, "rapidez da imagem em função de p\n(marque ‘conferir’)",
                 ha="center", va="center", transform=ax2.transAxes, color=CINZA)
        return

    ps = np.linspace(f + 0.6, 105, 500)
    qdot = (q_lente(ps, f) / ps) ** 2 * abs(pdot)   # q' = (q/p)² |p'|  (positivo = se afasta)
    ax2.plot(ps, qdot, color=VERMELHO, lw=2, label="rapidez da imagem |q′|")
    ax2.axhline(abs(pdot), color=AZUL, ls="--", label="rapidez do objeto |p′| = 5 cm/s")
    ax2.axvline(f, color=CINZA, ls=":")
    ax2.set_yscale("log")
    ax2.set_xlim(f - 2, 105)
    ax2.set_ylim(0.1, 4000)
    ax2.set_ylabel("cm/s")
    qd_atual = (q / p) ** 2 * abs(pdot)
    ax2.plot(p, qd_atual, "o", color=VERMELHO, ms=9)
    ax2.set_title(f"p = {br(p, 0)} cm: a imagem se afasta a {br(qd_atual, 2)} cm/s")
    # destaques das respostas
    ax2.plot(60, 1.25, "s", color=VERDE, ms=8)
    ax2.annotate("(a) p = 60: 1,25 cm/s", (60, 1.25), xytext=(63, 0.35), color=VERDE,
                 arrowprops=dict(arrowstyle="-", color=VERDE))
    ax2.plot(40, 5, "s", color=VERDE, ms=8)
    ax2.annotate("(b) p = 2f = 40 cm", (40, 5), xytext=(50, 12), color=VERDE,
                 arrowprops=dict(arrowstyle="-", color=VERDE))
    ax2.text(f + 1, 1500, "(c) p → f:\nq′ → ∞", color=CINZA, va="top")
    ax2.legend(loc="upper right", fontsize=8.5)


def plot_lente(p=60.0, conferir=False):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    desenhar_lente(axs, p, conferir)
    fig.tight_layout()
    plt.show()


interact(
    plot_lente,
    p=FloatSlider(min=25, max=100, step=1, value=60, description="p (cm)"),
    conferir=False,
);

interactive(children=(FloatSlider(value=60.0, description='p (cm)', min=25.0, step=1.0), Checkbox(value=False,…

In [6]:
def animar_lente(conferir=False, n_frames=60):
    """O objeto se aproxima a 5 cm/s (p de 100 a 25 cm) e vemos a imagem se afastar."""
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    fig.tight_layout()
    ps = np.linspace(100.0, 25.0, n_frames)

    def update(i):
        desenhar_lente(axs, ps[i], conferir)
        axs[0].set_title(f"t = {br((100.0 - ps[i]) / abs(V_OBJ), 1)} s   |   1/p + 1/q = 1/f")

    mostrar_animacao(fig, update, n_frames)


interact_manual(animar_lente, conferir=False);

interactive(children=(Checkbox(value=False, description='conferir'), IntSlider(value=60, description='n_frames…

<details>
<summary><b>📋 Resolução do Problema 1 (professor)</b></summary>

**Relação (vale em todo instante):** $\dfrac1p+\dfrac1q=\dfrac1f$, com $f=20$ cm constante.

**(a)** Derivando em $t$ (cadeia: $\frac{d}{dt}\frac1p=-\frac{p'}{p^2}$):

$$-\frac{p'}{p^2}-\frac{q'}{q^2}=0 \;\Longrightarrow\; q'=-\frac{q^2}{p^2}\,p'$$

*Só agora* substituímos. Em $p=60$: $\frac1q=\frac1{20}-\frac1{60}=\frac{2}{60}$, logo $q=30$ cm. O objeto se aproxima, então $p'=-5$ cm/s:

$$q'=-\frac{900}{3600}\,(-5)=+1{,}25\ \text{cm/s}$$

O sinal positivo indica que $q$ aumenta: a imagem **se afasta** da lente.

**(b)** $|q'|=|p'|\Rightarrow \dfrac{q^2}{p^2}=1\Rightarrow q=p$ (imagem real, $q>0$). Então $\frac2p=\frac1f\Rightarrow p=2f=40$ cm.
Para $p>2f$ a imagem é mais lenta que o objeto; para $f<p<2f$ é mais rápida.

**(c)** Conferindo por outra rota: $q=\dfrac{fp}{p-f}$ e, pela regra do quociente,

$$q'=-\frac{f^2\,p'}{(p-f)^2}=\frac{5f^2}{(p-f)^2}$$

Em $p=60$: $\frac{5\cdot400}{1600}=1{,}25$ ✓. Quando $p\to f$, o denominador vai a $0$ e $q'\to\infty$: é o objeto no foco, com a imagem no infinito. O modelo ideal deixa de valer ali, porque os raios saem paralelos.

</details>

---
### Problema 2 — Braço articulado &nbsp; ✏️ *(para os alunos)*

Dois segmentos de $3$ m e $4$ m formam um ângulo $\theta$ que aumenta a $0{,}2$ rad/s. Seja $d$ a distância entre as pontas livres.

- **(a)** Com que rapidez $d$ varia quando $\theta = 60^\circ$?
- **(b)** Em que ângulo $d$ deixa de crescer?

*Contexto:* é o modelo mais simples de um braço robótico ou de um mecanismo articulado: dois elos presos numa junta, e a distância entre as pontas depende do ângulo da junta.

**Na figura:** arraste $\theta$ e observe $d(\theta)$. A resposta só aparece ao marcar **conferir**.

<details>
<summary><b>💡 Dicas</b></summary>

1. Qual teorema liga os três lados de um triângulo a um dos ângulos? Escreva $d^2$ em função de $\theta$.
2. Derive os dois lados em $t$: no lado esquerdo você precisa da regra da cadeia em $d^2$.
3. Cuidado com as unidades: $\theta$ está em **radianos** (a taxa é $0{,}2$ rad/s).
4. Para (b): quando uma taxa "deixa de crescer", o que vale para a derivada?

</details>

In [ ]:
L1, L2 = 3.0, 4.0   # comprimentos dos segmentos (m)
W_ANG = 0.2         # θ' (rad/s)


def dist_braco(theta):
    """Distância entre as pontas livres (lei dos cossenos)."""
    return np.sqrt(L1**2 + L2**2 - 2 * L1 * L2 * np.cos(theta))


def ddist_braco(theta, w=W_ANG):
    """d' em m/s:  d' = L1·L2·sen(θ)·θ' / d."""
    return L1 * L2 * np.sin(theta) * w / dist_braco(theta)


def desenhar_braco(axs, theta_graus, conferir=False):
    ax1, ax2, ax3 = axs
    for ax in axs:
        ax.clear()
    th = np.radians(theta_graus)
    A = np.array([L1, 0.0])
    B = L2 * np.array([np.cos(th), np.sin(th)])
    d = dist_braco(th)

    # --- geometria do braço ---
    ax1.plot([0, A[0]], [0, 0], color=AZUL, lw=5, solid_capstyle="round")
    ax1.plot([0, B[0]], [0, B[1]], color=VERMELHO, lw=5, solid_capstyle="round")
    ax1.plot([A[0], B[0]], [A[1], B[1]], color=CINZA, ls="--", lw=2)
    ax1.plot(0, 0, "o", color=CINZA, ms=8)
    ax1.plot(*A, "o", color=AZUL, ms=9)
    ax1.plot(*B, "o", color=VERMELHO, ms=9)
    ax1.add_patch(Arc((0, 0), 1.6, 1.6, theta1=0, theta2=theta_graus, color=CINZA, lw=1.5))
    ax1.text(1.15 * np.cos(th / 2), 1.15 * np.sin(th / 2), "θ", ha="center", va="center", fontsize=13)
    mid = (A + B) / 2
    ax1.text(mid[0] + 0.15, mid[1] + 0.2, "d", color=CINZA, fontsize=14)
    ax1.text(L1 / 2, -0.55, "3 m", color=AZUL, ha="center")
    ax1.text(B[0] / 2 - 0.35 * np.sin(th), B[1] / 2 + 0.35 * np.cos(th), "4 m", color=VERMELHO, ha="center")
    ax1.set_xlim(-4.6, 4.6)
    ax1.set_ylim(-1.1, 4.8)
    ax1.set_aspect("equal")
    ax1.axis("off")
    ax1.set_title(f"θ = {br(theta_graus, 0)}°   →   d = {br(d, 2)} m")

    # --- d(θ) ---
    ths = np.linspace(0, 180, 361)
    ax2.plot(ths, dist_braco(np.radians(ths)), color=CINZA, lw=2)
    ax2.plot(theta_graus, d, "o", color=VERMELHO, ms=9)
    ax2.set_xticks(range(0, 181, 30))
    ax2.set_xlabel("θ (graus)")
    ax2.set_ylabel("d (m)")
    ax2.set_title("distância entre as pontas")

    # --- d'(θ): só com "conferir" ---
    ax3.set_xlabel("θ (graus)")
    if not conferir:
        ax3.set_xticks([])
        ax3.set_yticks([])
        ax3.grid(False)
        ax3.text(0.5, 0.5, "d′(θ) = ?\n(marque ‘conferir’)", ha="center", va="center",
                 transform=ax3.transAxes, color=CINZA)
        ax3.set_title("rapidez de variação de d")
        return
    ax3.plot(ths, ddist_braco(np.radians(ths)), color=VERMELHO, lw=2)
    ax3.axhline(0, color=CINZA, lw=0.8)
    ax3.plot(theta_graus, ddist_braco(th), "o", color=VERMELHO, ms=9)
    ax3.plot(60, ddist_braco(np.radians(60)), "s", color=VERDE, ms=8)
    ax3.annotate(f"(a) θ = 60°: {br(ddist_braco(np.radians(60)), 2)} m/s", (60, ddist_braco(np.radians(60))),
                 xytext=(95, 0.27), color=VERDE, arrowprops=dict(arrowstyle="-", color=VERDE))
    ax3.plot(180, 0, "s", color=VERDE, ms=8)
    ax3.annotate("(b) θ = 180°: d′ = 0", (180, 0), xytext=(100, 0.1), color=VERDE,
                 arrowprops=dict(arrowstyle="-", color=VERDE))
    ax3.set_xticks(range(0, 181, 30))
    ax3.set_ylabel("d′ (m/s)")
    ax3.set_title(f"d′ = {br(ddist_braco(th), 2)} m/s")


def plot_braco(theta_graus=60, conferir=False):
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
    desenhar_braco(axs, theta_graus, conferir)
    fig.tight_layout()
    plt.show()


interact(
    plot_braco,
    theta_graus=FloatSlider(min=5, max=180, step=1, value=60, description="θ (graus)"),
    conferir=False,
);

interactive(children=(FloatSlider(value=60.0, description='θ (graus)', max=180.0, min=5.0, step=1.0), Checkbox…

In [ ]:
def animar_braco(conferir=False, n_frames=72):
    """O ângulo abre de 5° a 180° (a 0,2 rad/s) e acompanhamos d."""
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
    fig.tight_layout()
    angulos = np.linspace(5, 180, n_frames)

    def update(i):
        desenhar_braco(axs, angulos[i], conferir)

    mostrar_animacao(fig, update, n_frames)


interact_manual(animar_braco, conferir=False);

interactive(children=(Checkbox(value=False, description='conferir'), IntSlider(value=72, description='n_frames…

<details>
<summary><b>✏️ Resolução do Problema 2 (gabarito)</b></summary>

**Relação (lei dos cossenos):** $d^2=3^2+4^2-2\cdot3\cdot4\cos\theta=25-24\cos\theta$.

**(a)** Derivando em $t$:

$$2d\,d'=24\sin\theta\;\theta' \;\Longrightarrow\; d'=\frac{12\,\sin\theta\;\theta'}{d}$$

Em $\theta=60^\circ$: $d^2=25-12=13$, então $d=\sqrt{13}\approx3{,}61$ m. Com $\theta'=0{,}2$ rad/s:

$$d'=\frac{12\cdot\frac{\sqrt3}{2}\cdot0{,}2}{\sqrt{13}}\approx0{,}58\ \text{m/s}$$

**(b)** $d'=0\Rightarrow\sin\theta=0$. Com $\theta$ crescendo a partir de $0$, isso ocorre em $\theta=180^\circ$ (braço esticado, $d=7$ m). Antes disso, $\sin\theta>0$, então $d'>0$: $d$ cresce até o máximo.

**🔎 Desafio extra:** em que ângulo $d'$ é *máxima*? Resposta: $\cos\theta=\frac34$ ($\theta\approx41{,}4^\circ$), onde $d'=0{,}6$ m/s — exatamente a rapidez da ponta do segmento de $3$ m ($3\cdot0{,}2$). Nesse ângulo, $d$ é perpendicular ao segmento de $3$ m. Dá para ver o pico no gráfico de $d'(\theta)$ com **conferir**.

</details>

---
### Problema 3 — Cometa e lei das áreas &nbsp; 📋 *(no quadro)*

Um cometa obedece a $r^2\dot\theta = h$ (constante). Em $r = 3$ UA, $\dot\theta = 0{,}4$ rad/ano e $\dot r = 2$ UA/ano.

- **(a)** Calcule $dA/dt$.
- **(b)** Derive $r^2\dot\theta = h$ e obtenha $\ddot\theta$.
- **(c)** Calcule $\dfrac{d(v_\theta)}{dt}$, com $v_\theta = r\dot\theta$, de duas formas: regra do produto e $v_\theta = h/r$.

*Contexto:* é a Segunda Lei de Kepler em ação — o gancho da aula. Aqui $\dot r$ é a velocidade **radial** (afastando do Sol) e $v_\theta=r\dot\theta$ a velocidade **transversal** (girando em torno do Sol).

**Na figura:** os dados do enunciado correspondem a uma órbita kepleriana real (veja a célula de verificação mais adiante). Os dois setores têm o mesmo $\Delta t$. Marque **conferir** depois de resolver.

In [9]:
GM_SOL = 4 * np.pi**2   # UA³/ano²  (unidades: UA e anos)


def orbita_do_enunciado(r0=3.0, rdot0=2.0, thdot0=0.4):
    """Órbita kepleriana compatível com os dados do enunciado (só para visualizar:
    a resolução NÃO precisa disto, só de r²θ̇ = h). Supõe ṙ > 0 (cometa se afastando)."""
    h = r0**2 * thdot0                              # lei das áreas
    v2 = rdot0**2 + (r0 * thdot0) ** 2              # |v|² = ṙ² + (rθ̇)²
    a = 1 / (2 / r0 - v2 / GM_SOL)                  # equação vis-viva
    e = np.sqrt(1 - h**2 / (GM_SOL * a))            # h² = GM·a·(1 − e²)
    T = a**1.5                                      # 3ª lei de Kepler (UA, anos)
    t_grid = np.linspace(0, T / 2, 20001)           # ramo em que o cometa se afasta (ṙ > 0)
    x, y = orbit_xy(t_grid, a, e, T)
    t_star = np.interp(r0, np.hypot(x, y), t_grid)  # instante em que r = r0
    return dict(h=h, a=a, e=e, T=T, t_star=t_star)


COMETA = orbita_do_enunciado()


def polar_cometa(t, dt=1e-4):
    """r, ṙ e θ̇ do cometa no instante t (derivadas numéricas por diferença central)."""
    def rth(tt):
        x, y = orbit_xy(np.asarray(tt, dtype=float), COMETA["a"], COMETA["e"], COMETA["T"])
        return np.hypot(x, y), np.arctan2(y, x)

    r0_, _ = rth(t)
    rp, thp = rth(t + dt)
    rm, thm = rth(t - dt)
    return float(r0_), float((rp - rm) / (2 * dt)), float((thp - thm) / (2 * dt))


def desenhar_cometa(axs, conferir=False, dt_setor=0.1):
    ax1, ax2 = axs
    ax1.clear()
    ax2.clear()
    a, e, T, ts = COMETA["a"], COMETA["e"], COMETA["T"], COMETA["t_star"]

    # --- órbita, Sol e setores varridos ---
    E = np.linspace(0, 2 * np.pi, 600)
    ax1.plot(a * (np.cos(E) - e), a * np.sqrt(1 - e**2) * np.sin(E), color=CINZA, lw=1.2)
    ax1.plot(0, 0, "o", color=LARANJA, ms=14)
    for t0, cor, rotulo in ((ts, AZUL, "setor em r = 3 UA"), (0.0, VERMELHO, "setor perto do periélio")):
        tt = np.linspace(t0, t0 + dt_setor, 80)
        xs, ys = orbit_xy(tt, a, e, T)
        ax1.fill(np.concatenate([[0.0], xs]), np.concatenate([[0.0], ys]),
                 color=cor, alpha=0.55, label=f"{rotulo} (Δt = 0,1 ano)")

    # --- cometa e componentes da velocidade ---
    xc, yc = (float(v) for v in orbit_xy(np.asarray(ts), a, e, T))
    r, rdot, thdot = polar_cometa(ts)
    rhat = np.array([xc, yc]) / r
    that = np.array([-yc, xc]) / r
    k = 0.35  # escala visual das setas
    ax1.plot(xc, yc, "o", color=CINZA, ms=8, zorder=5)
    ax1.plot([0, xc], [0, yc], color=CINZA, lw=1, ls="--")
    ax1.annotate("", xy=(xc, yc) + k * rdot * rhat, xytext=(xc, yc),
                 arrowprops=dict(arrowstyle="-|>", color=VERMELHO, lw=2.5, mutation_scale=14))
    ax1.annotate("", xy=(xc, yc) + k * r * thdot * that, xytext=(xc, yc),
                 arrowprops=dict(arrowstyle="-|>", color=VERDE, lw=2.5, mutation_scale=14))
    p1 = (xc, yc) + k * rdot * rhat
    p2 = (xc, yc) + k * r * thdot * that
    ax1.text(p1[0] - 0.1, p1[1] + 0.12, r"$\dot{r}$", color=VERMELHO, fontsize=14, ha="center")
    ax1.text(p2[0] + 0.05, p2[1] - 0.1, r"$r\dot{\theta}$", color=VERDE, fontsize=14, va="top")
    ax1.text(xc / 2, yc / 2 + 0.12, "r", color=CINZA, fontsize=12)
    ax1.set_xlim(-4.3, 1.2)
    ax1.set_ylim(-1.9, 1.9)
    ax1.set_aspect("equal")
    ax1.grid(False)
    ax1.legend(loc="lower left", fontsize=8.5)
    ax1.set_title(r"$r$ = 3 UA,  $\dot{r}$ = 2 UA/ano,  $\dot{\theta}$ = 0,4 rad/ano", fontsize=11)

    if conferir:
        A1 = swept_area(ts, ts + dt_setor, a, e, T)
        A2 = swept_area(0.0, dt_setor, a, e, T)
        ax1.text(0.98, 0.97,
                 f"A ≈ {br(A1, 3)} e {br(A2, 3)} UA²\n"
                 f"dA/dt = A/Δt ≈ {br(A1 / dt_setor, 2)} UA²/ano",
                 transform=ax1.transAxes, ha="right", va="top", fontsize=9.5)

    # --- θ̇(t) e v_θ(t): só com "conferir" ---
    ax2.set_xlabel("t − t* (anos)")
    if not conferir:
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.grid(False)
        ax2.text(0.5, 0.5, r"$\dot{\theta}(t)$ e $v_\theta(t)$" + "\n(marque ‘conferir’)",
                 ha="center", va="center", transform=ax2.transAxes, color=CINZA)
        return

    tl = np.linspace(ts - 0.35, ts + 0.35, 201)
    x, y = orbit_xy(tl, a, e, T)
    rr = np.hypot(x, y)
    thd = np.gradient(np.unwrap(np.arctan2(y, x)), tl)
    ax2.plot(tl - ts, thd, color=AZUL, lw=2, label=r"$\dot{\theta}$ (rad/ano)")
    ax2.plot(tl - ts, rr * thd, color=LARANJA, lw=2, label=r"$v_\theta = r\dot{\theta}$ (UA/ano)")
    thdd = derivada(lambda t: polar_cometa(t)[2], ts, h=1e-3)
    vthd = derivada(lambda t: polar_cometa(t)[0] * polar_cometa(t)[2], ts, h=1e-3)
    s = np.array([-0.3, 0.3])   # retas tangentes (inclinação = a resposta)
    ax2.plot(s, thdot + thdd * s, "--", color="black", lw=1.3, zorder=4)
    ax2.plot(s, r * thdot + vthd * s, "--", color="black", lw=1.3, zorder=4)
    ax2.plot(0, thdot, "o", color=AZUL)
    ax2.plot(0, r * thdot, "o", color=LARANJA)
    ax2.text(0.04, 0.62, r"$\ddot{\theta}$ ≈ " + br(thdd, 3) + " rad/ano²", color=AZUL, fontsize=10)
    ax2.text(0.04, 1.30, r"$v_\theta'$ ≈ " + br(vthd, 2) + " UA/ano²", color=LARANJA, fontsize=10)
    ax2.set_ylim(0, 2.0)
    ax2.legend(loc="lower left", fontsize=9)
    ax2.set_title("as duas diminuem (tracejado = reta tangente)")


def plot_cometa(conferir=False):
    fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.15, 1]})
    desenhar_cometa(axs, conferir)
    fig.tight_layout()
    plt.show()


interact(plot_cometa, conferir=False);

interactive(children=(Checkbox(value=False, description='conferir'), Output()), _dom_classes=('widget-interact…

<details>
<summary><b>📋 Resolução do Problema 3 (professor)</b></summary>

**(a)** Pela ponte da Parte 1: $\dfrac{dA}{dt}=\dfrac12r^2\dot\theta=\dfrac h2$. Aqui $h=r^2\dot\theta=9\cdot0{,}4=3{,}6$, logo

$$\frac{dA}{dt}=\frac{3{,}6}{2}=1{,}8\ \text{UA}^2/\text{ano}$$

**(b)** Derive $r^2\dot\theta=h$ ($h$ constante), com produto e cadeia:

$$2r\dot r\,\dot\theta+r^2\ddot\theta=0 \;\Longrightarrow\; \ddot\theta=-\frac{2\dot r\dot\theta}{r}=-\frac{2\cdot2\cdot0{,}4}{3}\approx-0{,}533\ \text{rad/ano}^2$$

O sinal negativo faz sentido: o cometa se afasta ($\dot r>0$) e gira cada vez mais devagar.

**(c)** *Rota 1 (produto):*

$$v_\theta'=\dot r\dot\theta+r\ddot\theta=2\cdot0{,}4+3\cdot(-0{,}533)=0{,}8-1{,}6=-0{,}8\ \text{UA/ano}^2$$

*Rota 2 ($v_\theta=h/r$):*

$$v_\theta'=-\frac{h\,\dot r}{r^2}=-\frac{3{,}6\cdot2}{9}=-0{,}8\ \text{UA/ano}^2\ ✓$$

As duas rotas coincidem, o que confirma o $\ddot\theta$ do item (b).

*Nota:* $h=r^2\dot\theta$ é o **momento angular por unidade de massa**; que ele seja constante é o que a lei das áreas diz.

</details>

---
### Problema 4 — Decaimento orbital &nbsp; ✏️ *(para os alunos)*

Um satélite em órbita circular de raio $a = 6900$ km perde $2$ km de altitude por dia. Pela 3ª lei de Kepler, $T = 2\pi\sqrt{a^3/\mu}$, com $\mu = 3{,}986\cdot10^{5}\ \text{km}^3/\text{s}^2$.

- **(a)** Qual é a variação relativa $T'/T$ por dia?
- **(b)** Em quantos segundos por dia o período diminui?

*Contexto:* em órbita baixa ainda há um pouquinho de atmosfera, e o arrasto vai roubando energia do satélite. Com $R_\oplus\approx6371$ km, esse satélite está a cerca de $529$ km de altitude.

**Na figura:** à esquerda, a 3ª lei de Kepler como função $T(a)$; à direita, uma extrapolação (irreal!) do que aconteceria se a perda fosse sempre $2$ km/dia. Marque **conferir** para ver a resposta.

<details>
<summary><b>💡 Dicas</b></summary>

1. Escreva $T=2\pi\mu^{-1/2}\,a^{3/2}$: o que é constante aqui? ($\mu$ e $2\pi$.)
2. Para a **variação relativa** $T'/T$ não precisa calcular $T$ antes. Tome o logaritmo: $\ln T=\ln2\pi-\frac12\ln\mu+\frac32\ln a$ e derive em $t$.
3. A taxa $a'$ está em km/**dia**; $T'$ sai em s/**dia**.

</details>

In [ ]:
MU = 3.986e5            # km³/s²
R_TERRA = 6371.0        # km
A0, A_DOT = 6900.0, -2.0  # raio inicial (km) e taxa a' (km/dia)


def periodo(a):
    """3ª lei de Kepler para órbita circular: T = 2π √(a³/μ), em segundos (a em km)."""
    return 2 * np.pi * np.sqrt(a**3 / MU)


def desenhar_satelite(axs, conferir=False):
    ax1, ax2 = axs
    ax1.clear()
    ax2.clear()
    T0 = periodo(A0)

    # --- T(a) ---
    a_vals = np.linspace(R_TERRA, 7400, 300)
    ax1.plot(a_vals, periodo(a_vals), color=CINZA, lw=2)
    ax1.axvline(R_TERRA, color=VERMELHO, ls=":")
    ax1.text(R_TERRA + 10, 6200, "superfície\n(a = R⊕)", color=VERMELHO, va="top")
    ax1.plot(A0, T0, "o", color=AZUL, ms=9)
    ax1.text(A0 + 20, T0 - 120, f"a = 6900 km\nT ≈ {br(T0 / 60, 1)} min", color=AZUL, va="top")
    ax1.set_xlabel("raio da órbita a (km)")
    ax1.set_ylabel("período T (s)")
    ax1.set_title("3ª lei de Kepler:  T = 2π√(a³/μ)")
    if conferir:
        inclin = 1.5 * T0 / A0   # dT/da = 3T/(2a)  (s/km)
        xs = np.array([A0 - 250, A0 + 250])
        ax1.plot(xs, T0 + inclin * (xs - A0), "--", color=AZUL, lw=1.5)
        ax1.text(0.97, 0.05, f"inclinação dT/da = 3T/(2a) ≈ {br(inclin, 3)} s/km",
                 transform=ax1.transAxes, color=AZUL, ha="right", va="bottom", fontsize=10)

    # --- extrapolação: e se a taxa fosse sempre -2 km/dia? ---
    dias = np.linspace(0, (A0 - R_TERRA) / abs(A_DOT), 200)   # até a = R_Terra
    ax2.plot(dias, periodo(A0 + A_DOT * dias), color=AZUL, lw=2)
    ax2.set_xlabel("tempo (dias)")
    ax2.set_ylabel("período T (s)")
    ax2.set_title("e se a perda fosse sempre 2 km/dia?")
    ax2.text(0.97, 0.97, "irreal: o arrasto cresce\nquando a altitude cai",
             transform=ax2.transAxes, ha="right", va="top", color=CINZA, fontsize=9.5)
    if conferir:
        rel = 1.5 * A_DOT / A0          # T'/T = (3/2) a'/a  (por dia)
        ax2.text(0.03, 0.05,
                 f"T′/T = (3/2)·a′/a ≈ {br(rel * 1e4, 2)}·10⁻⁴ por dia  ({br(rel * 100, 3)} %)\n"
                 f"T′ ≈ {br(rel * T0, 1)} s por dia",
                 transform=ax2.transAxes, ha="left", va="bottom", color=VERDE, fontsize=10)


def plot_satelite(conferir=False):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    desenhar_satelite(axs, conferir)
    fig.tight_layout()
    plt.show()


interact(plot_satelite, conferir=False);

interactive(children=(Checkbox(value=False, description='conferir'), Output()), _dom_classes=('widget-interact…

<details>
<summary><b>✏️ Resolução do Problema 4 (gabarito)</b></summary>

**(a)** Reescreva $T=2\pi\mu^{-1/2}a^{3/2}$ e aplique $\ln$:

$$\ln T=\ln2\pi-\tfrac12\ln\mu+\tfrac32\ln a$$

Derivando em $t$ ($\mu$ e $2\pi$ são constantes):

$$\frac{T'}{T}=\frac32\,\frac{a'}{a}=\frac32\cdot\frac{-2}{6900}\approx-4{,}35\cdot10^{-4}\ \text{por dia}\quad(\approx-0{,}043\,\%)$$

Não foi preciso calcular $T$ para obter a variação relativa. *(Conferência pela regra da potência: $T'=2\pi\mu^{-1/2}\cdot\frac32a^{1/2}a'$; dividindo por $T$ dá o mesmo resultado.)*

**(b)** Agora calcule $T$:

$$T=2\pi\sqrt{\frac{6900^3}{3{,}986\cdot10^5}}\approx5704\ \text{s}\ (\approx95\ \text{min})$$

$$T'=\left(-4{,}35\cdot10^{-4}\right)(5704)\approx-2{,}5\ \text{s por dia}$$

**Para pensar:** (i) se a perda fosse sempre $2$ km/dia, o satélite chegaria à superfície em $\frac{529}{2}\approx264$ dias — mas o arrasto *aumenta* quando a altitude cai, então a taxa real não é constante: $a'$ é uma taxa **instantânea**. (ii) A velocidade orbital é $v=\sqrt{\mu/a}$, então $\frac{v'}{v}=-\frac12\frac{a'}{a}\approx+1{,}45\cdot10^{-4}$ por dia: um satélite que perde altitude por arrasto **acelera**.

</details>

---
## Parte 4 — Conferindo tudo com Python

A **derivada numérica** $\dfrac{f(t+h)-f(t-h)}{2h}$ é a definição de derivada sem o limite: com $h$ bem pequeno, o computador devolve a taxa de variação sem usar nenhuma regra de derivação. Se a resolução à mão e o computador concordam, a modelagem está certa — e é um bom hábito para quem vai programar simulações.

A célula abaixo refaz todas as respostas assim: monta $p(t)$, $\theta(t)$, $a(t)$ etc. e deriva numericamente.

In [11]:
def conferir_numero(rotulo, obtido, esperado, unidade="", nd=3, tol=1e-2):
    ok = abs(obtido - esperado) <= tol * max(1.0, abs(esperado))
    print(f"{'✓' if ok else '✗'} {rotulo:<44} numérico {br(obtido, nd):>9}   à mão {br(esperado, nd):>9}  {unidade}")


print("Problema 1 — lente")
# p(t) = 60 − 5t  ->  q(p(t)) derivado em t = 0
conferir_numero("(a) q′ em p = 60", derivada(lambda t: q_lente(60 + V_OBJ * t), 0.0), 1.25, "cm/s")
ps = np.arange(21, 100, 0.01)
qdots = np.array([derivada(lambda t, p=p: q_lente(p + V_OBJ * t), 0.0) for p in ps])
conferir_numero("(b) p em que |q′| = |p′|", ps[np.argmin(np.abs(qdots - abs(V_OBJ)))], 40.0, "cm", nd=2)

print("\nProblema 2 — braço")
conferir_numero("(a) d′ em θ = 60°", derivada(lambda t: dist_braco(np.radians(60) + W_ANG * t), 0.0),
                12 * np.sin(np.radians(60)) * 0.2 / np.sqrt(13), "m/s")
conferir_numero("(b) d′ perto de θ = 180°", derivada(lambda t: dist_braco(np.radians(179.99) + W_ANG * t), 0.0),
                0.0, "m/s", nd=4)

print("\nProblema 3 — cometa (órbita kepleriana compatível com os dados)")
t_s = COMETA["t_star"]
r_n, rdot_n, thdot_n = polar_cometa(t_s)
conferir_numero("dados: r em t*", r_n, 3.0, "UA")
conferir_numero("dados: ṙ em t*", rdot_n, 2.0, "UA/ano")
conferir_numero("dados: θ̇ em t*", thdot_n, 0.4, "rad/ano")
dt_s = 0.01
conferir_numero("(a) dA/dt", swept_area(t_s, t_s + dt_s, COMETA["a"], COMETA["e"], COMETA["T"]) / dt_s, 1.8, "UA²/ano")
conferir_numero("(b) θ̈", derivada(lambda t: polar_cometa(t)[2], t_s, h=1e-3), -2 * 2 * 0.4 / 3, "rad/ano²")
conferir_numero("(c) v_θ′", derivada(lambda t: polar_cometa(t)[0] * polar_cometa(t)[2], t_s, h=1e-3), -0.8, "UA/ano²")
print(f"   (a órbita compatível tem a ≈ {br(COMETA['a'])} UA, e ≈ {br(COMETA['e'])}, T ≈ {br(COMETA['T'])} anos)")

print("\nProblema 4 — decaimento orbital")
T_num = derivada(lambda t: periodo(A0 + A_DOT * t), 0.0)                  # s por dia
conferir_numero("(a) T′/T", T_num / periodo(A0) * 1e4, 1.5 * A_DOT / A0 * 1e4, "×10⁻⁴ por dia")
conferir_numero("(b) T′", T_num, 1.5 * A_DOT / A0 * periodo(A0), "s por dia")

Problema 1 — lente
✓ (a) q′ em p = 60                             numérico     1,250   à mão     1,250  cm/s
✓ (b) p em que |q′| = |p′|                     numérico     40,00   à mão     40,00  cm

Problema 2 — braço
✓ (a) d′ em θ = 60°                            numérico     0,576   à mão     0,576  m/s
✓ (b) d′ perto de θ = 180°                     numérico    0,0001   à mão    0,0000  m/s

Problema 3 — cometa (órbita kepleriana compatível com os dados)
✓ dados: r em t*                               numérico     3,000   à mão     3,000  UA
✓ dados: ṙ em t*                               numérico     2,000   à mão     2,000  UA/ano
✓ dados: θ̇ em t*                              numérico     0,400   à mão     0,400  rad/ano
✓ (a) dA/dt                                    numérico     1,800   à mão     1,800  UA²/ano
✓ (b) θ̈                                       numérico    −0,533   à mão    −0,533  rad/ano²
✓ (c) v_θ′                                     numérico    −0,800   à mão    −0,

---
## Fechamento — o que levar para a prova

- **Método:** nomear → relacionar → **derivar em $t$** → só então substituir → interpretar (sinal e unidade).
- **Armadilhas:** substituir antes de derivar; esquecer a regra da cadeia; misturar graus e radianos; confundir taxa **instantânea** com taxa constante.
- **Conferência:** resolver por duas rotas (como no Problema 3) ou checar com derivada numérica (Parte 4).
- **Conexão com a história:** a lei das áreas de Kepler é uma taxa relacionada que Newton provou ser constante — e o mesmo raciocínio que descreve um cometa descreve uma lente, um braço robótico e um satélite.

### Para ir além

- Problema 2: provar que $d'$ é máxima quando $\cos\theta=L_1/L_2$ e vale $L_1\theta'$.
- Problema 3: mudar `r0`, `rdot0` e `thdot0` em `orbita_do_enunciado()` e ver como a órbita muda.
- Problema 4: incluir um arrasto que cresce quando a altitude cai, e deixar de supor $a'$ constante.

### Referências

- J. Kepler, *Astronomia Nova* (1609) — origem da lei das áreas.
- I. Newton, *Philosophiæ Naturalis Principia Mathematica* (1687), Livro I, Proposição 1 — a lei das áreas para qualquer força central.